# Validate against buoy observations

Compare **offshore hindcast** (closest goal node), **bulk metamodel**, **spectral metamodel**, and **dynamic SnapWave** to buoy records.

**Prerequisites**

1. Fitted models — `06_Validate_Metamodel.ipynb` (`mm.fit_all`)
2. Dynamic run for the same period — `08_Dynamic_Cases_Generation.ipynb`
3. Observations GeoJSON — `03_Generate_Observations_Configuration.ipynb` → `inputs/Observations/<Case>_observations.json`
4. Processed buoy file(s) under `inputs/Observations/processed/` (you build these in the notebook below)

Use the **same `period`** for buoy data, reconstruction, and dynamic output.

Comparisons request **`hs`, `tp`, `dir`, and `spr`**. Variables missing from the buoy
file or any model are skipped automatically.

The buoys validated are the `ok` buoy sites of `outputs/output_sites_{model}.csv` (notebook 03); keep a subset in the configuration cell if needed.

In [ ]:
import matplotlib.pyplot as plt
import time
import xarray as xr
from utils.config import resolve
from utils.geometry import attach_site_meta
from utils import simulation as sim
from utils import validation as val
from utils import metamodel as mm
from utils import onshore_sites as obs

## Configuration

Pick buoy sites from the observations GeoJSON (`instrument_id` keys). SnapWave site
labels are `buoy_{instrument_id}`. Point each site to its processed NetCDF (bulk or
spectral) under `inputs/Observations/processed/`.

In [ ]:
period = slice("2014-10-23", "2022-06-01")
offshore_name = "WHACS"
mda_n = 450  # must match a fitted mda_candidates entry
mm_set = "buoys"  # metamodel set (metamodel.sets) fitted on the buoys

# Modelled buoys (ok output sites, notebook 03); keep a subset here if needed
sites = obs.list_snapwave_buoy_sites()

# Processed observation NetCDF per site: inputs/Observations/processed/<instrument_id>.nc
buoy_raw_files = {
    site: resolve(f"inputs/Observations/processed/{obs.instrument_id(site)}.nc")
    for site in sites
}

requested_variables = ["hs", "tp", "dir", "spr"]

dynamic_path = (
    sim.cases_dir("dynamic") / f"Dynamic_Cases_{offshore_name}_2020.nc"
)

print("Observations JSON:", obs.geojson_path())
print(f"period         : {period}")
print(f"mda_n          : {mda_n}")
print(f"metamodel      : {mm_set}")
print(f"sites          : {sites}")
print(f"requested vars : {requested_variables}")
print(f"dynamic path   : {dynamic_path}")
print(f"dynamic exists : {dynamic_path.is_file()}")

## 1 · Check models

In [ ]:
mm.require_models(mm_set, mda_n)

## 2 · Process buoy observations

Load each processed file, rename variables to HyGloW names (`hs`, `tp`, `dir`, `spr`),
slice to `period`, and build a validation dataset on `(time, sites)` with CF-style
attributes. Site metadata (lon/lat/depth) comes from the observations GeoJSON.

In [ ]:
# CF-style attributes for validation plots
_BUOY_VAR_ATTRS = {
    "hs": {"standard_name": "sea_surface_wave_significant_height", "units": "m"},
    "tp": {"standard_name": "sea_surface_wave_period", "units": "s"},
    "dir": {"standard_name": "sea_surface_wave_from_direction", "units": "degree"},
    "spr": {
        "standard_name": "sea_surface_wave_directional_spread",
        "units": "degree",
    },
}

chunks: list[xr.Dataset] = []
for site_id in sites:
    raw = xr.open_dataset(resolve(buoy_raw_files[site_id])).sel(time=period)

    if "Efth" in raw or "efth" in raw:
        spec_name = "Efth" if "Efth" in raw else "efth"
        stats = raw[spec_name].spec.stats(["hs", "tp", "dm", "dspr"]).load()
        site_ds = stats.rename({"dm": "dir", "dspr": "spr"})
    else:
        rename = {}
        if "dspr" in raw.data_vars and "spr" not in raw.data_vars:
            rename["dspr"] = "spr"
        if rename:
            raw = raw.rename(rename)
        vars_out = {}
        for name in requested_variables:
            if name in raw.data_vars:
                vars_out[name] = raw[name]
            elif name == "dir" and "dm" in raw.data_vars:
                vars_out["dir"] = raw["dm"]
        if "hs" not in vars_out:
            raise ValueError(f"{site_id}: processed buoy file must contain hs or Efth")
        site_ds = xr.Dataset(vars_out)

    site_ds = site_ds.expand_dims(sites=[site_id]).transpose("time", "sites")
    for var, attrs in _BUOY_VAR_ATTRS.items():
        if var in site_ds:
            site_ds[var].attrs.update(attrs)
    chunks.append(site_ds)

buoy = attach_site_meta(xr.concat(chunks, dim="sites"))
buoy

## 3 · Reconstruct metamodel, load dynamic, and closest-goal offshore hindcast

Metamodel bulk/spectral reconstructions, dynamic SnapWave, and bulk offshore forcing
at each buoy's **closest goal hindcast node** (all sources in the case JSON —
currently WHACS only).

In [ ]:
# Resources for the reconstruction below (at most 75% of this machine).
# Edit res if wanted (res.workers = ...), then run the next cell.
res = mm.reconstruction_resources(
    period, mda_n, metamodel=mm_set, spectral=True, offshore_name=offshore_name, sites=sites,
)

In [ ]:
timings: dict[str, float] = {}

t0 = time.perf_counter()
bulk = mm.reconstruct(
    period,
    mda_n,
    metamodel=mm_set,
    spectral=False,
    offshore_name=offshore_name,
    sites=sites,
    resources=res,
)
timings["Bulk metamodel"] = time.perf_counter() - t0

t0 = time.perf_counter()
spectral = mm.reconstruct(
    period,
    mda_n,
    metamodel=mm_set,
    spectral=True,
    offshore_name=offshore_name,
    sites=sites,
    resources=res,
)
timings["Spectral metamodel"] = time.perf_counter() - t0
# Near-zero above means these sites/period were already cached from a
# previous run, not a fresh-reconstruction cost. Offshore hindcast below
# is a plain lookup, not a reconstruction, so it has no timing entry.

if not dynamic_path.is_file():
    raise FileNotFoundError(
        f"Dynamic output not found: {dynamic_path}\n"
        "Run 08_Dynamic_Cases_Generation for this period or change `period`."
    )

dynamic = xr.open_dataset(dynamic_path).sel(time=period)
dynamic = dynamic.sel(sites=bulk.sites.values).load()

offshore_hindcast, closest_goals = val.load_offshore_hindcast_at_buoys(
    period,
    sites,
    variables=requested_variables,
)
closest_goals

estimates = {
    **offshore_hindcast,
    "Bulk metamodel": bulk,
    "Spectral metamodel": spectral,
    val.DYNAMIC_REFERENCE_LABEL: dynamic,
}

variables = val.resolve_comparison_variables(
    buoy,
    *estimates.values(),
    variables=requested_variables,
)
print(f"Comparing: {variables}")

## 4 · Metrics vs buoy

In [ ]:
results = val.compare_all(
    buoy,
    estimates,
    variables=variables,
    reference_label="Buoy",
)

summary = val.metrics_summary(results)
summary

## 4b · Save summary

In [ ]:
# `period` is a slice here (not a single string) — derive a clean,
# filesystem-safe label from it once; the summary PNG save cell reuses it.
period_label = period if isinstance(period, str) else f"{period.start}-{period.stop}"

summary_out = summary.copy()
summary_out.insert(0, "period", period_label)
summary_out.insert(1, "metamodel", mm_set)
summary_out.insert(2, "mda_candidate", mda_n)
summary_out.insert(3, "offshore_name", offshore_name)
summary_out["reconstruct_seconds"] = summary_out["model"].map(timings)

out_path = val.buoy_validation_path("onshore", period_label)
out_path.parent.mkdir(parents=True, exist_ok=True)
summary_out.to_csv(out_path, index=False)
print(f"Wrote {out_path}")

## 5 · Scatter plots

In [ ]:
for site in sites:
    for name, result in results.items():
        result.plot_scatter(
            site,
            variables=variables,
            estimate_label=name,
            reference_label="Buoy",
        )
        plt.suptitle(f"{name} vs buoy — {site} — {period}")
        plt.show()

## 6 · Time series

Line plots of buoy observations vs each model over the validation period (direction as scatter).

In [ ]:
for site in sites:
    val.plot_buoy_timeseries(buoy, estimates, site, variables=variables)
    plt.suptitle(f"Buoy validation — {site} — {period}")
    plt.show()

## 7 · Save per-buoy summary PNG (GUI)

Density scatter (top row, buoy vs. spectral metamodel) + one wide Hs/Tp time series (bottom row, cropped to when the buoy actually has data) — one combined PNG per buoy, shown as-is in the GUI's Buoy Validation tab when you click that buoy's star on the map.

In [ ]:
for site in sites:
    fig = val.plot_buoy_summary(
        buoy,
        estimates["Spectral metamodel"],
        site,
        variables=variables,
        reference_label="Buoy",
        period_label=period_label,
    )
    out_path = val.buoy_summary_figure_path(site)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"saved {out_path}")